# Kaggle DKT Training Pipeline

> **IMPORTANT KAGGLE SETUP:**
> Configure your Kaggle Notebook settings before running:
> 1. Set Accelerator to **GPU T4x2**.
> 2. Add the Riiid dataset using `Add Data` -> Search "Riiid Answer Correctness Prediction"
>
> *(Note: We use Kaggle's standard input dataset paths in the code below. If you use a custom path locally, please update the input paths appropriately)*.

In [ ]:
import os
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

def process_data(input_csv="/kaggle/input/riiid-test-answer-prediction/train.csv", output_dir="/kaggle/working/data/processed", max_seq_len=100, max_rows=500000):
    print(f"Loading data from {input_csv} (max {max_rows} rows)...")
    # Load dataset
    df = pd.read_csv(input_csv, nrows=max_rows)
    
    # Riiid dataset specific: filter out lectures (content_type_id == 1)
    if 'content_type_id' in df.columns:
        df = df[df.content_type_id == 0]
        
    df = df[['user_id', 'content_id', 'answered_correctly']]
    df.columns = ['user_id', 'question_id', 'answered_correctly']
    
    print("Encoding question IDs...")
    # Encode question_ids as integers
    q_unique = df['question_id'].unique()
    q_mapping = {q: i + 1 for i, q in enumerate(q_unique)} # 0 reserved for padding
    df['question_id'] = df['question_id'].map(q_mapping)
    
    print("Grouping into sequences...")
    # Group by user_id
    grouped = df.groupby('user_id').agg({
        'question_id': list,
        'answered_correctly': list
    })
    
    # Convert lists to tensors
    sequences_q = [torch.tensor(q, dtype=torch.long) for q in grouped['question_id']]
    sequences_a = [torch.tensor(a, dtype=torch.float32) for a in grouped['answered_correctly']]
    
    print(f"Padding sequences to length {max_seq_len}...")
    # Truncate sequences that are too long
    sequences_q = [q[-max_seq_len:] for q in sequences_q]
    sequences_a = [a[-max_seq_len:] for a in sequences_a]
    
    # Pad sequences
    padded_q = pad_sequence(sequences_q, batch_first=True, padding_value=0)
    padded_a = pad_sequence(sequences_a, batch_first=True, padding_value=-1) # -1 for padding in answers
    
    # If padding didn't reach max_seq_len for all, add explicit padding
    if padded_q.size(1) < max_seq_len:
        pad_size = max_seq_len - padded_q.size(1)
        padded_q = torch.cat([padded_q, torch.zeros(padded_q.size(0), pad_size, dtype=torch.long)], dim=1)
        padded_a = torch.cat([padded_a, torch.full((padded_a.size(0), pad_size), -1, dtype=torch.float32)], dim=1)
    
    print("Splitting dataset...")
    # Split 80 / 10 / 10
    total_users = padded_q.size(0)
    indices = list(range(total_users))
    
    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)
    
    print("Saving tensors...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Save splits
    torch.save({
        'q': padded_q[train_idx],
        'a': padded_a[train_idx]
    }, os.path.join(output_dir, "train.pt"))
    
    torch.save({
        'q': padded_q[val_idx],
        'a': padded_a[val_idx]
    }, os.path.join(output_dir, "val.pt"))
    
    torch.save({
        'q': padded_q[test_idx],
        'a': padded_a[test_idx]
    }, os.path.join(output_dir, "test.pt"))
    
    print(f"Preprocessing complete. Saved {len(train_idx)} train, {len(val_idx)} val, {len(test_idx)} test sequences.")
    print(f"Unique questions encoded: {len(q_unique)}")

# Execution for Kaggle cell
process_data()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score

# ========================
# 1. MODEL ARCHITECTURE
# ========================
class DKT(nn.Module):
    def __init__(self, n_questions, embed_size=128, hidden_size=128, num_layers=2):
        super(DKT, self).__init__()
        self.n_questions = n_questions
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Embeddings for each interaction: 0 pad, 1 to n (wrong), n+1 to 2n (correct)
        self.embedding = nn.Embedding(2 * n_questions + 1, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, n_questions + 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, q_seq, a_seq):
        interaction = q_seq + self.n_questions * a_seq.clamp(min=0).long()
        interaction = interaction.masked_fill(q_seq == 0, 0)
        embeds = self.embedding(interaction)
        out, _ = self.lstm(embeds)
        logits = self.fc(out)
        return self.sigmoid(logits)


# ========================
# 2. TRAINING LOOP
# ========================
def train_dkt():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device} | This training step takes 3–5 hours typically.")

    # Hyperparameters
    batch_size = 256
    n_epochs = 20
    lr = 3e-4
    weight_decay = 1e-2
    patience = 3
    data_dir = "/kaggle/working/data/processed"

    print("Loading data...")
    train_data = torch.load(os.path.join(data_dir, "train.pt"), map_location="cpu", weights_only=True)
    val_data = torch.load(os.path.join(data_dir, "val.pt"), map_location="cpu", weights_only=True)

    n_questions = max(train_data['q'].max().item(), val_data['q'].max().item())
    print(f"Num questions encoded: {n_questions}")

    train_dataset = TensorDataset(train_data['q'], train_data['a'])
    val_dataset = TensorDataset(val_data['q'], val_data['a'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = DKT(n_questions=n_questions).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    criterion = nn.BCELoss(reduction='none')
    scaler = GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

    best_val_auc = 0.0
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        
        for q_batch, a_batch in train_loader:
            q_batch, a_batch = q_batch.to(device), a_batch.to(device)
            optimizer.zero_grad()
            
            with autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                preds = model(q_batch, a_batch)
                q_target = q_batch[:, 1:].clone().long()
                a_target = a_batch[:, 1:].clone()
                preds_subset = preds[:, :-1, :]
                mask = (a_target != -1)
                
                gathered_preds = torch.gather(preds_subset, 2, q_target.unsqueeze(2)).squeeze(2)
                loss_unreduced = criterion(gathered_preds[mask], a_target[mask])
                loss = loss_unreduced.mean()
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * q_batch.size(0)
            
        train_loss = total_loss / len(train_dataset)

        # Validation
        model.eval()
        val_preds_all, val_targets_all = [], []
        
        with torch.no_grad():
            for q_batch, a_batch in val_loader:
                q_batch, a_batch = q_batch.to(device), a_batch.to(device)
                preds = model(q_batch, a_batch)
                q_target, a_target = q_batch[:, 1:].clone().long(), a_batch[:, 1:].clone()
                preds_subset = preds[:, :-1, :]
                mask = (a_target != -1)
                
                gathered_preds = torch.gather(preds_subset, 2, q_target.unsqueeze(2)).squeeze(2)
                val_preds_all.extend(gathered_preds[mask].cpu().numpy())
                val_targets_all.extend(a_target[mask].cpu().numpy())
                
        val_auc = roc_auc_score(val_targets_all, val_preds_all)
        print(f"Epoch {epoch+1:02d}/{n_epochs} | Loss: {train_loss:.4f} | Val AUC-ROC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "dkt_best.pt")
            epochs_no_improve = 0
            print(f"  --> Saved new best model: dkt_best.pt (AUC: {best_val_auc:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  --> No improvement. Patience: {epochs_no_improve}/{patience}")
            if epochs_no_improve >= patience:
                print("Early stopping triggered.")
                break

# Execute Training
train_dkt()

In [ ]:
from IPython.display import FileLink

# When the DKT model training is fully completed (3-5 hours expected), 
# this cell will generate a link to download the model checkpoint file over your browser.
FileLink(r'dkt_best.pt')